# Generate explanations
For every test user, the target item is the user's top-1 recommendation. Each explainer scores the items in the user's history, and the ranked list is saved as an explanation dictionary:

`data/<dataset>/<recommender>_<explainer>_expl_dict.pkl` → `{user_id: [(history_item, score), ...]}`, most important first.

Explainers produced here: Jaccard, Cosine (similarity-based), LIME, SHAP (post-hoc), ACCENT and LXR (counterfactual). SPINRec explanations are produced with the authors' code and saved in the same format (see README).

Set `data_name`, `recommender_name` and `explainers_to_run` below, then run all cells.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import pickle
import importlib
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
import ipynb

from config import *

In [ ]:
data_name = "ML1M"          # ML1M, Yahoo, Pinterest
recommender_name = "MLP"    # MLP (= MF in the paper), VAE
explainers_to_run = ['jaccard', 'cosine', 'lime', 'shap', 'accent', 'lxr']

files_path = dataset_dir(data_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

output_type = OUTPUT_TYPE[recommender_name]
num_items = NUM_ITEMS[data_name]
num_features = NUM_ITEMS[data_name]
hidden_dim = HIDDEN_DIM[(data_name, recommender_name)]
recommender_path = CHECKPOINTS_DIR / RECOMMENDER_CHECKPOINTS[(data_name, recommender_name)]
lxr_path, lxr_dim = LXR_CHECKPOINTS[(data_name, recommender_name)]

## Data

In [ ]:
test_data = pd.read_csv(files_path / f'test_data_{data_name}.csv', index_col=0)
static_test_data = pd.read_csv(files_path / f'static_test_data_{data_name}.csv', index_col=0)
test_array = static_test_data.iloc[:,:-2].to_numpy()
items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)

kw_dict = {'device':device,
          'num_items': num_items,
          'num_features': num_items,
          'demographic':False,
          'all_items_tensor':all_items_tensor,
          'static_test_data':static_test_data,
          'items_array':items_array,
          'output_type':output_type,
          'recommender_name':recommender_name}

In [ ]:
# Precomputed inputs for the similarity-based explainers and SHAP
if 'jaccard' in explainers_to_run or 'cosine' in explainers_to_run:
    with open(files_path / f'jaccard_based_sim_{data_name}.pkl', 'rb') as f:
        jaccard_dict = pickle.load(f)
    with open(files_path / f'cosine_based_sim_{data_name}.pkl', 'rb') as f:
        cosine_dict = pickle.load(f)
    for i in range(num_items):
        for j in range(i, num_items):
            jaccard_dict[(j,i)]= jaccard_dict[(i,j)]
            cosine_dict[(j,i)]= cosine_dict[(i,j)]

if 'shap' in explainers_to_run:
    with open(files_path / f'item_to_cluster_{recommender_name}_{data_name}.pkl', 'rb') as f:
        item_to_cluster = pickle.load(f)
    with open(files_path / f'shap_values_{recommender_name}_{data_name}.pkl', 'rb') as f:
        shap_values = pickle.load(f)

## Recommender and LXR explainer

In [ ]:
from ipynb.fs.defs.help_functions import *
from ipynb.fs.defs.recommenders_architecture import *
from ipynb.fs.defs.lime import *
lime = LimeBase(distance_to_proximity)

In [ ]:
def load_recommender():
    if recommender_name=='MLP':
        recommender = MLP(hidden_dim, **kw_dict)
    elif recommender_name=='VAE':
        if data_name == "Pinterest":
            recommender = VAE(PINTEREST_VAE_CONFIG, **kw_dict)
        else:
            recommender = VAE(VAE_CONFIG, **kw_dict)
    else:
        raise ValueError(f"Unknown recommender: {recommender_name}")

    recommender_checkpoint = torch.load(recommender_path, map_location=torch.device('cpu'))
    recommender.load_state_dict(recommender_checkpoint)
    recommender.eval()
    for param in recommender.parameters():
        param.requires_grad= False
    return recommender

recommender = load_recommender()

In [ ]:
class Explainer(nn.Module):
    def __init__(self, user_size, item_size, hidden_size):
        super(Explainer, self).__init__()
        
        self.users_fc = nn.Linear(in_features = user_size, out_features=hidden_size).to(device)
        self.items_fc = nn.Linear(in_features = item_size, out_features=hidden_size).to(device)
        self.bottleneck = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features = hidden_size*2, out_features=hidden_size).to(device),
            nn.Tanh(),
            nn.Linear(in_features = hidden_size, out_features=user_size).to(device),
            nn.Sigmoid()
        ).to(device)
        
        
    def forward(self, user_tensor, item_tensor):
        user_output = self.users_fc(user_tensor.float())
        item_output = self.items_fc(item_tensor.float())
        combined_output = torch.cat((user_output, item_output), dim=-1)
        expl_scores = self.bottleneck(combined_output).to(device)

        return expl_scores

In [ ]:
def load_explainer():
    explainer = Explainer(num_features, num_items, lxr_dim)
    lxr_checkpoint = torch.load(CHECKPOINTS_DIR / lxr_path, map_location=torch.device('cpu'))
    explainer.load_state_dict(lxr_checkpoint)
    explainer.eval()
    for param in explainer.parameters():
        param.requires_grad= False
    return explainer

if 'lxr' in explainers_to_run:
    lxr_explainer = load_explainer()

## Explainers
Each function returns `{history_item: importance}` for the items in the user's history.

In [ ]:
#User based similarities using Jaccard
def find_jaccard_mask(x, item_id, user_based_Jaccard_sim):
    user_hist = x # remove the positive item we want to explain from the user history
    user_hist[item_id] = 0
    item_jaccard_dict = {}
    for i,j in enumerate(user_hist>0):
        if j:
            if (i,item_id) in user_based_Jaccard_sim:
                item_jaccard_dict[i]=user_based_Jaccard_sim[(i,item_id)] # add Jaccard similarity between items
            else:
                item_jaccard_dict[i] = 0            

    return item_jaccard_dict

In [ ]:
#Cosine based similarities between users and items
def find_cosine_mask(x, item_id, item_cosine):
    user_hist = x # remove the positive item we want to explain from the user history
    user_hist[item_id] = 0
    item_cosine_dict = {}
    for i,j in enumerate(user_hist>0):
        if j:
            if (i,item_id) in item_cosine:
                item_cosine_dict[i]=item_cosine[(i,item_id)]
            else:
                item_cosine_dict[i]=0

    return item_cosine_dict

In [ ]:
def find_lime_mask(x, item_id, min_pert, max_pert, num_of_perturbations, kernel_func, feature_selection, recommender, num_samples=10, method = 'POS', **kw_dict):
    user_hist = x # remove the positive item we want to explain from the user history
    user_hist[item_id] = 0
    lime.kernel_fn = kernel_func
    neighborhood_data, neighborhood_labels, distances, item_id = get_lime_args(user_hist, item_id, recommender, all_items_tensor, min_pert = min_pert, max_pert = max_pert, num_of_perturbations = num_of_perturbations, seed = item_id, **kw_dict)
    if method=='POS':
        most_pop_items  = lime.explain_instance_with_data(neighborhood_data, neighborhood_labels, distances, item_id, num_samples, feature_selection, pos_neg='POS')
    if method=='NEG':
        most_pop_items  = lime.explain_instance_with_data(neighborhood_data, neighborhood_labels, distances, item_id, num_samples, feature_selection ,pos_neg='NEG')
        
    return most_pop_items

In [ ]:
def find_shapley_mask(user_tensor, user_id, model, shap_values, item_to_cluster):
    item_shap = {}
    shapley_values = shap_values[shap_values[:, 0].astype(int) == user_id][:,1:]
    user_vector = user_tensor.cpu().detach().numpy().astype(int)

    for i in np.where(user_vector.astype(int) == 1)[0]:
        items_cluster = item_to_cluster[i]
        item_shap[i] = shapley_values.T[int(items_cluster)][0]

    return item_shap

In [ ]:
def find_fia_mask(user_tensor, item_tensor, item_id, recommender):
    y_pred = recommender_run(user_tensor, recommender, item_tensor, item_id, **kw_dict).to(device)
    items_fia = {}
    user_hist = user_tensor.cpu().detach().numpy().astype(int)
    
    for i in range(num_items):
        if(user_hist[i] == 1):
            user_hist[i] = 0
            user_tensor = torch.FloatTensor(user_hist).to(device)
            y_pred_without_item = recommender_run(user_tensor, recommender, item_tensor, item_id, 'single', **kw_dict).to(device)
            infl_score = y_pred - y_pred_without_item
            items_fia[i] = infl_score
            user_hist[i] = 1

    return items_fia

In [ ]:
def find_accent_mask(user_tensor, user_id, item_tensor, item_id, recommender_model, top_k):
   
    items_accent = defaultdict(float)
    factor = top_k - 1
    user_accent_hist = user_tensor.cpu().detach().numpy().astype(int)

    #Get topk items
    sorted_indices = list(get_top_k(user_tensor, user_tensor, recommender_model, **kw_dict).keys())
    
    if top_k == 1:
        # When k=1, return the index of the first maximum value
        top_k_indices = [sorted_indices[0]]
    else:
        top_k_indices = sorted_indices[:top_k]
   

    for iteration, item_k_id in enumerate(top_k_indices):

        # Set topk items to 0 in the user's history
        user_accent_hist[item_k_id] = 0
        user_tensor = torch.FloatTensor(user_accent_hist).to(device)
       
        item_vector = items_array[item_k_id]
        item_tensor = torch.FloatTensor(item_vector).to(device)
              
        # Check influence of the items in the history on this specific item in topk
        fia_dict = find_fia_mask(user_tensor, item_tensor, item_k_id, recommender_model)
         
        # Sum up all differences between influence on top1 and other topk values
        if not iteration:
            for key in fia_dict.keys():
                items_accent[key] *= factor
        else:
            for key in fia_dict.keys():
                items_accent[key] -= fia_dict[key]
       
    for key in items_accent.keys():
        items_accent[key] *= -1    

    return items_accent

In [ ]:
def find_lxr_mask(x, item_tensor, explainer):
    
    user_hist = x
    expl_scores = explainer(user_hist, item_tensor)
    x_masked = user_hist*expl_scores
    item_sim_dict = {}
    for i,j in enumerate(x_masked!=0):
        if j:
            item_sim_dict[i]=x_masked[i] 
        
    return item_sim_dict

In [ ]:
def single_user_expl(user_vector, user_tensor, item_id, item_tensor, num_items, recommender_model, user_id = None, mask_type = None):
    '''
    Runs one explainer for one user and returns a list of (history_item, score),
    sorted by score (most important first).
    '''
    user_hist_size = np.sum(user_vector)

    if mask_type == 'lime':
        POS_sim_items = find_lime_mask(user_vector, item_id, 50, 100, 150, distance_to_proximity,'highest_weights', recommender_model, num_samples=user_hist_size, **kw_dict)
    else:
        if mask_type == 'jaccard':
            sim_items = find_jaccard_mask(user_tensor, item_id, jaccard_dict)
        elif mask_type == 'cosine':
            sim_items = find_cosine_mask(user_tensor, item_id, cosine_dict)
        elif mask_type == 'shap':
            sim_items = find_shapley_mask(user_tensor, user_id, recommender_model, shap_values, item_to_cluster)
        elif mask_type == 'accent':
            sim_items = find_accent_mask(user_tensor, user_id, item_tensor, item_id, recommender_model, 5)
        elif mask_type == 'lxr':
            sim_items = find_lxr_mask(user_tensor, item_tensor, lxr_explainer)

        POS_sim_items = list(sorted(sim_items.items(), key=lambda item: item[1],reverse=True))[0:user_hist_size]

    return POS_sim_items

## Generate and save explanation dictionaries

In [ ]:
expl_dicts = {name: {} for name in explainers_to_run}

recommender.eval()
with torch.no_grad():
    for i in tqdm(range(test_array.shape[0])):
        user_vector = test_array[i]
        user_tensor = torch.FloatTensor(user_vector).to(device)
        user_id = int(test_data.index[i])

        item_id = int(get_user_recommended_item(user_tensor, recommender, **kw_dict).detach().cpu().numpy())
        item_vector =  items_array[item_id]
        item_tensor = torch.FloatTensor(item_vector).to(device)

        user_vector[item_id] = 0
        user_tensor[item_id] = 0

        for name in explainers_to_run:
            expl_dicts[name][user_id] = single_user_expl(user_vector, user_tensor, item_id, item_tensor, num_items, recommender,
                                                         mask_type=name, user_id=user_id if name == 'shap' else None)

for name, expl_dict in expl_dicts.items():
    with open(expl_dict_path(data_name, recommender_name, name), 'wb') as handle:
        pickle.dump(expl_dict, handle)
    print(f'saved {expl_dict_path(data_name, recommender_name, name)}')